## Generet marker genelists from databases

In [1]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

Load library

In [2]:
# Path-related libraries
import os
import glob
from pyhere import here  # For reproducible relative paths
import sys # system specific parameters
from pathlib import Path # File system paths

# Numerical operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd

# Download from internet
import requests # http

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities
import misc as misc   

Parameters

In [3]:
# create directory for marker genes
ma.create_directories(here('data/marker_database'))
ma.create_directories(here('data/marker_database/harmonized'))
markers_dir = Path(here('data/marker_database'))
harmo_dir = Path(here('data/marker_database/harmonized'))

/work/islet_cartography_scrna/data/marker_database Directory already exists!
/work/islet_cartography_scrna/data/marker_database/harmonized Directory already exists!


Cell rename dictionary

In [4]:
cell_type_map = {
    # --- PANCREAS EXOCRINE ---
    "acinar_cell": [
        "acinar",
        "Acinar cell", 
        "Acinar cells", 
    ],
    "exocrine_general": ["Exocrine cell"],
    "ductal_cell": [
        "ductal",
        "Ductal cell", 
        "Ductal cells", 
        "Ductal epithelial cell", 
        "Epithelial ductal cell", 
        "Pancreatic ductal cell", 
        "Pancreatic ductal stem cell", 
        "TFF1+ ductal cell", 
        "TUBA1A+ ductal cell"
    ],

    # --- PANCREAS ENDOCRINE ---
    "alpha_cell": [
        "alpha",
        "Alpha cell", 
        "Alpha cell (α cell)", 
        "Alpha cells"
    ],
    "beta_cell": [
        "beta",
        "Beta cell", 
        "Beta cell(β cell)", 
        "Beta cells", 
        "Beta-like cell"
    ],
    "delta_cell": [
        "delta",
        "Delta cell", 
        "Delta cells", 
        "δ cell"
    ],
    "gamma_cell": [
        "gamma",
        "Gamma (PP) cells", 
        "Gamma cell", 
        "PP cell", 
        "Pancreatic polypeptide cell"
    ],
    "epsilon_cell": [
        "Epsilon cell",
        "epsilon"
    ],
    "endocrine_general": [
        "Endocrine cell", 
        "Endocrine progenitor cell", 
        "Enteroendocrine cell", 
        "Immature endocrine cell", 
        "Neuroendocrine cell", 
        "SCGB2A1+ endocrine cell"
    ],

    # --- IMMUNE: LYMPHOID (T, B, NK, ILC) ---
    "t_cell": [
        "CD4 T cell", 
        "CD8 T cell", 
        "Cycling T/NK cell", 
        "Cytotoxic Lymphocyte", 
        "Gamma delta T cells",
        "GZMB CD8 T cell", 
        "KLRB1 cytotoxic CD4 T", 
        "Memory CD8 T cell", 
        "Naive CD4 T cell", 
        "Naive T cell", 
        "Non-recirculating tissue-resident memory T cell", 
        "Regulatory T (Treg) cell", 
        "Regulatory T(Treg) cell", 
        "T cell", 
        "T cells", 
        "T cytotoxic cells", 
        "T helper 1(Th1) cell", 
        "T helper 2(Th2) cell", 
        "T helper cell", 
        "T memory cells", 
        "T/NK cell", 
        "Tfh cell", 
        "abT (entry) cell"
    ],
    "b_cell_lineage": [
        "B cell", 
        "B cell precursor", 
        "B cells", 
        "B cells naive", 
        "Cycling B", 
        "Follicular B cell", 
        "Plasma cell", 
        "Plasma cells"
    ],
    "nk_cell": [
        "CD16 NK cell", 
        "NK cells", 
        "Natural killer T(NKT) cell", 
        "Natural killer cell"
    ],
    "ilc": [
        "ILC"
    ],
    "lymphoid_general": [
        "Lymphocyte", 
        "Lymphoid cell"
    ],

    # --- IMMUNE: MYELOID ---
    "macrophage": [
        "Alveolar macrophage", 
        "LYVE1 macrophage", 
        "M1 macrophage", 
        "Macrophage", 
        "Macrophages", 
        "Microglia", 
        "Pro-inflammatory macrophage", 
        "Tissue resident macrophage"
    ],
    "monocyte": [
        "Monocyte", 
        "Monocyte/DC", 
        "Monocytes"
    ],
    "dendritic_cell": [
        "Dendritic cell", 
        "Dendritic cells", 
        "Plasmacytoid dendritic cells", 
        "cDC", 
        "cDC2", 
        "mregDC"
    ],
    "mast_cell": [
        "Mast cell", 
        "Mast cells"
    ],
    "neutrophil": [
        "Neutrophils"
    ],
    "myeloid": [
        "Cycling myeloid cell", 
        "Myeloid cell"
    ],
    "immune": ["immune", "Immune cell"],

    # --- STROMAL & MESENCHYMAL ---
    "fibroblast": [
        "APOE+ fibroblast", 
        "CCL2+SFRP2+ fibroblast", 
        "CFD+MGP+ fibroblast", 
        "CXCL+ fibroblast", 
        "CXCL1/2/3 fibroblast", 
        "Cycling fibroblast", 
        "EMT-like cell", 
        "Fibroblast", 
        "GREB1+ fibroblast", 
        "IGFBP6+SFRP4+ fibroblast", 
        "MMP1+ fibroblast", 
        "Mesenchymal cell", 
        "Metallothionein+ Collagen- fibroblast", 
        "Myofibroblast", 
        "PLAC9+ fibroblast", 
        "POSTN fibroblast", 
        "RNASE1+ fibroblast", 
        "TNC fibroblast"
    ],
    "stellate_cell": [
        "Pancreatic stellate cell", 
        "Pancreatic stellate cells", 
        "Stellate cell"
    ],
    "activated_stellate_cell": ["activated_stellate_cell"],
    "quiescent_stellate_cell": ["quiescent_stellate_cell"],
    "pericyte": [
        "CXCL+ pericyte", 
        "Pericyte", 
        "Pericytes", 
        "Perivascular cell"
    ],
    "smooth_muscle": [
        "CREB+MT1A+ vascular smooth muscle cell", 
        "Skeletal muscle myoblast", 
        "Smooth muscle cell", 
        "Vascular smooth muscle cell"
    ],
    "mesothelial_cell": [
        "Mesothelial cell"
    ],

    # --- ENDOTHELIAL ---
    "endothelial": ["endothelial", "Endothelial cell", "Endothelial cells", "Endothelial‑mesenchymal transition EC"],
    "endothelial_vascular": [
        "Arterial EC", 
        "Capillary EC", 
        "Fetal MHC- Collagen+ capillary EC", 
        "Venous EC"
    ],
    "endothelial_lymphatic": [
        "Lymphatic EC", 
        "Lymphatic endothelial cell"
    ],

    # --- EPITHELIAL (NON-PANCREAS) ---
    "cholangiocyte": [
        "CXCL6+ cholangiocyte", 
        "SPINK1+SGMS2+ cholangiocyte", 
        "TFF+ cholangiocyte"
    ],
    "epithelial_intestinal_gastric": [
        "BEST4 enterocyte", 
        "CXCL2/3+ surface mucous cell", 
        "Enterocyte", 
        "Epithelial cell of intestine", 
        "Goblet cell", 
        "Paneth cell", 
        "Surface mucous cell", 
        "Tuft cell", 
        "Type B intercalated cell"
    ],
    "epithelial_airway": [
        "Airway basal cell", 
        "Club cell"
    ],
    "epithelial_mammary": [
        "KRT17 mammary luminal cell", 
        "Mammary luminal cell", 
        "PIP mammary luminal cell", 
        "SFN mammary luminal progenitor", 
        "Secretoglobin mammary luminal cell"
    ],
    "epithelial_urothelial_bladder": [
        "Basal cell of bladder", 
        "DKK1+MMP1+ basal cell of bladder", 
        "FOS+ basal cell of bladder", 
        "MMP7+ basal cell of bladder"
    ],
    "epithelial_other": [
        "Apical cell", 
        "Basal cell", 
        "CLDN3+ epithelial cell", 
        "Ciliary body cell", 
        "Epithelial cell", 
        "Epithelial cell of conjunctiva", 
        "Epithelial cell of ovary", 
        "Epithelial cell of pancreas", 
        "Metallothionein+ epithelial cell"
    ],

    # --- NEURAL & GLIAL ---
    "neuron": [
        "Cone-OFF-bipolar cell", 
        "Neuron", 
        "Splatter neuron", 
        "Sympathoblast", 
        "VIP inhibitory neuron"
    ],
    "glial_cell": [
        "Glial cell"],
    "astrocyte": ["Astrocyte"], 
    
    "schwann_cell": ["schwann", 
                     "Peri-islet Schwann cells", 
                     "Schwann cell"],
    "melanocyte": [
        "Eye melanocyte"
    ],

    # --- HEMATOPOIETIC & ERYTHROID ---
    "erythrocyte": [
        "Blood cell", 
        "Late hemoglobin+ erythroblast", 
        "Red blood cell"
    ],
    "hematopoietic_progenitor": [
        "Hematopoietic precursor cell", 
        "MPP"
    ],

    # --- STEM, PROGENITOR & CYCLING (GENERAL) ---
    "progenitor_general": [
        "Pancreatic progenitor cell", 
        "Pancreatic stem cell", 
        "Progenitor cell", 
        "Stem cell", 
        "Transit-amplifying cell", 
        "Undifferentiated pancreatic progenitor cell"
    ],
    "cycling_general": [
        "Proliferative cell",
        "cycling"
    ],

    # --- OTHER ---
    "cancer_associated": [
        "Cancer stem cell"
    ],
    "unknown": [
        "unknown"
    ]
}

# --- VERIFICATION ---

full_input_set = {
    # From first set
    'Acinar cells', 'Alpha cells', 'B cells', 'B cells naive', 'Beta cells', 'Delta cells',
    'Dendritic cells', 'Ductal cells', 'Endothelial cells', 'Gamma (PP) cells', 'Gamma delta T cells',
    'Macrophages', 'Mast cells', 'Monocytes', 'NK cells', 'Neutrophils', 'Pancreatic stellate cells',
    'Peri-islet Schwann cells', 'Pericytes', 'Plasma cells', 'Plasmacytoid dendritic cells', 'T cells',
    'T cytotoxic cells', 'T memory cells',

    # From second set
    'Acinar cell', 'Alpha cell (α cell)', 'Apical cell', 'B cell', 'Basal cell', 'Beta cell(β cell)',
    'Beta-like cell', 'Blood cell', 'Cancer stem cell', 'Cytotoxic Lymphocyte', 'Delta cell', 'Dendritic cell',
    'Ductal cell', 'Ductal epithelial cell', 'EMT-like cell', 'Endocrine cell', 'Endocrine progenitor cell',
    'Endothelial cell', 'Epithelial cell', 'Epithelial ductal cell', 'Exocrine cell', 'Fibroblast',
    'Follicular B cell', 'Gamma cell', 'Immature endocrine cell', 'Immune cell', 'Lymphatic endothelial cell',
    'Lymphocyte', 'Macrophage', 'Mast cell', 'Mesenchymal cell', 'Monocyte', 'Myeloid cell',
    'Natural killer T(NKT) cell', 'Natural killer cell', 'Neuroendocrine cell', 'Neuron',
    'Non-recirculating tissue-resident memory T cell', 'PP cell', 'Pancreatic ductal stem cell',
    'Pancreatic polypeptide cell', 'Pancreatic progenitor cell', 'Pancreatic stellate cell',
    'Pancreatic stem cell', 'Pro-inflammatory macrophage', 'Progenitor cell', 'Proliferative cell',
    'Regulatory T (Treg) cell', 'Regulatory T(Treg) cell', 'Schwann cell', 'Smooth muscle cell',
    'Stellate cell', 'Stem cell', 'T cell', 'T helper 1(Th1) cell', 'T helper 2(Th2) cell', 'T helper cell',
    'Undifferentiated pancreatic progenitor cell', 'δ cell',

    # From azimuth refernce annotation
    "Alpha cell", "Beta cell", "Macrophage", "PP cell", "Fibroblast", "Endothelial cell", "Delta cell",
    "Pericyte", "Acinar cell", "TUBA1A+ ductal cell", "unknown", "TFF1+ ductal cell", "Tissue resident macrophage",
    "Glial cell", "Myofibroblast", "Pancreatic ductal cell", "Epsilon cell", "Capillary EC", "Perivascular cell",
    "Metallothionein+ epithelial cell", "Epithelial cell", "mregDC", "Microglia", "MMP7+ basal cell of bladder",
    "Airway basal cell", "Vascular smooth muscle cell", "Eye melanocyte", "TFF+ cholangiocyte",
    "Epithelial cell of pancreas", "Cycling T/NK cell", "Tuft cell", "Type B intercalated cell",
    "Epithelial cell of conjunctiva", "CD8 T cell", "Goblet cell", "MMP1+ fibroblast", "Mast cell",
    "Enterocyte", "PIP mammary luminal cell", "Lymphatic EC", "Monocyte", "Naive CD4 T cell",
    "CD16 NK cell", "cDC2", "M1 macrophage", "SCGB2A1+ endocrine cell", "Enteroendocrine cell",
    "CXCL6+ cholangiocyte", "GREB1+ fibroblast", "DKK1+MMP1+ basal cell of bladder", "CLDN3+ epithelial cell",
    "VIP inhibitory neuron", "Plasma cell", "B cell precursor", "abT (entry) cell", "Red blood cell",
    "T cell", "Late hemoglobin+ erythroblast", "Mesothelial cell", "Paneth cell", "Hematopoietic precursor cell",
    "PLAC9+ fibroblast", "Club cell", "CFD+MGP+ fibroblast", "Venous EC", "CD4 T cell",
    "Endothelial‑mesenchymal transition EC", "Alveolar macrophage", "Cycling fibroblast", "Skeletal muscle myoblast",
    "T/NK cell", "B cell", "Monocyte/DC", "BEST4 enterocyte", "Transit-amplifying cell", "Memory CD8 T cell",
    "Epithelial cell of intestine", "Naive T cell", "Cycling B", "Lymphoid cell", "CXCL+ pericyte", "cDC",
    "IGFBP6+SFRP4+ fibroblast", "LYVE1 macrophage", "CXCL1/2/3 fibroblast", "CXCL+ fibroblast",
    "Secretoglobin mammary luminal cell", "CREB+MT1A+ vascular smooth muscle cell", "CCL2+SFRP2+ fibroblast",
    "Surface mucous cell", "Astrocyte", "APOE+ fibroblast", "Metallothionein+ Collagen- fibroblast",
    "Basal cell of bladder", "KLRB1 cytotoxic CD4 T", "FOS+ basal cell of bladder", "KRT17 mammary luminal cell",
    "Cycling myeloid cell", "Dendritic cell", "GZMB CD8 T cell", "SPINK1+SGMS2+ cholangiocyte", "Arterial EC",
    "TNC fibroblast", "ILC", "Cone-OFF-bipolar cell", "Neuron", "SFN mammary luminal progenitor",
    "Mammary luminal cell", "MPP", "POSTN fibroblast", "Fetal MHC- Collagen+ capillary EC",
    "Epithelial cell of ovary", "Tfh cell", "RNASE1+ fibroblast", "Sympathoblast", "CXCL2/3+ surface mucous cell",
    "Splatter neuron", "Ciliary body cell", "Endocrine cell"
}


# Flatten inputs
all_input_names = full_input_set

# Flatten map
mapped_names = set()
for names in cell_type_map.values():
    mapped_names.update(names)

# Find missing
missing = all_input_names - mapped_names

if not missing:
    print("Success: All names are accounted for.")
else:
    print(f"Missing names: {missing}")

# Reverse the map
reverse_map = {}
for canonical, aliases in cell_type_map.items():
    for a in aliases:
        reverse_map[misc.to_snake_case(a)] = canonical

# save as csv
rows = []
for old, new in reverse_map.items():
    rows.append({"old_cell_name": old, "new_cell_name": new})
        
df = pd.DataFrame(rows)
df.to_csv(os.path.join(markers_dir, 'cell_type_map.csv'), index=False)

Success: All names are accounted for.


### Download pancreatic cells marker genes lists

Panglaodb and cellmarker

In [5]:
# # Make directory
# data_dir = markers_dir

# # Define files and URLs (acess: 2025-10-14)(year/month/day)
# files = {
#     "panglaodb_markers.tsv.gz": "https://panglaodb.se/markers/PanglaoDB_markers_27_Mar_2020.tsv.gz",
#     "cellmarker20.xlsx": "http://www.bio-bigdata.center/CellMarker_download_files/file/Cell_marker_Human.xlsx"
# }

# for filename, url in files.items():
#     filepath = data_dir / filename
#     if not filepath.exists():
#         print(f"Downloading {filename}...")
#         r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, stream=True)
#         r.raise_for_status()
#         with open(filepath, "wb") as f:
#             for chunk in r.iter_content(chunk_size=8192):
#                 f.write(chunk)
#     else:
#         print(f"{filename} already exists, skipping download.")


Azimuth

In [6]:
# Azimuth marker genes https://azimuth.hubmapconsortium.org/references/#Human%20-%20Pancreas (acess: 2025-10-14) 
# manually written as there is no download
azimuth_genes = {
    "cycling_general": ["UBE2C","TOP2A","CDK1","BIRC5","PBK","CDKN3","MKI67","CDC20","CCNB2","CDCA3"],
    "immune": ["ACP5","APOE","HLA-DRA","TYROBP","LAPTM5","SDS","FCER1G","C1QC","C1QB","SRGN"],
    "quiescent_stellate_cell": ["RGS5","C11orf96","FABP4","CSRP2","IL24","ADIRF","NDUFA4L2","GPX3","IGFBP4","ESAM"],
    "endothelial": ["PLVAP","RGCC","ENG","PECAM1","ESM1","SERPINE1","CLDN5","STC1","MMP1","GNG11"],
    "schwann_cell": ["NGFR","CDH19","UCN2","SOX10","S100A1","PLP1","TSPAN11","WNT16","SOX2","TFAP2A"],
    "activated_stellate_cell": ["COL1A1","COL1A2","COL6A3","COL3A1","TIMP3","TIMP1","CTHRC1","SFRP2","BGN","LUM"],
    "epsilon_cell": ["BHMT","VSTM2L","PHGR1","TM4SF5","ANXA13","ASGR1","DEFB1","GHRL","COL22A1","OLFML3"],
    "gamma_cell": ["PPY","AQP3","MEIS2","ID2","GPC5-AS1","CARTPT","PRSS23","ETV1","PPY2","TUBB2A"],
    "delta_cell": ["SST","RBP4","SERPINA1","RGS2","PCSK1","SEC11C","HHEX","LEPR","MDK","LY6H"],
    "ductal_cell": ["SPP1","MMP7","IGFBP7","KRT7","ANXA4","SERPINA1","LCN2","CFTR","KRT19","SERPING1"],
    "acinar_cell": ["REG1A","PRSS1","CTRB2","CTRB1","REG1B","CELA3A","PRSS2","REG3A","CPA1","CLPS"],
    "beta_cell": ["IAPP","INS","DLK1","INS-IGF2","G6PC2","HADH","ADCYAP1","GSN","NPTX2","C12orf75"],
    "alpha_cell": ["GCG","TTR","PPP1R1A","CRYBA2","TM4SF4","MAFB","GC","GPX3","PCSK2","PEMT"]
}

TF database

Description from website:

> Here, we developed the TF-Marker database (TF-Marker, http://bio.liclab.net/TF-Marker/) which is committed to a comprehensive manual curation of TFs and related markers with experimental evidence in specific cell and tissue types in human. Currently, through reviewing 2,091 published literature, we have manually classified TFs and related markers into five types according to their functions: 1) TF: TFs, which regulate the expression of markers; 2) T Marker: markers, which are regulated by TFs (TF and T Marker pairs can identify cell types more specifically); 3) I Marker: markers, which influence the activity of TFs (I Markers can also influence the development of specific cells and tissues); 4) TFMarker: TFs, which play roles as markers (TFMarkers are cell/tissue-specific TFs used as cell markers in biology experiments); and 5) TF Pmarker: TFs, which play roles as potential markers. By curating thousands of published literature, 5,905 entries including 1,316 TFs, 1,092 T Markers, 473 I Markers, 1,600 TFMarkers and 1,424 TF Pmarkers, were annotated in 383 cell types and 95 tissue types in human. Moreover, TF-Marker divided markers into disease markers and tissue/cell-specific markers. TF-Marker is an elaborate database, which provides TFs and related markers supported by experimental evidence. We believe TF-Marker will provide strong support for research into cell/tissue-specific TFs and related markers.


Description of the types of markers in this database:

- **Type 1 markers (TF):** TFs which regulate expression of the markers
- **Type 2 (T marker):** Markers which are regulated by the TF
- **Type 3 (I marker):** marker which influence the activity of the TF
- **Type 5 (TF Pmarker):** TFs which play roles as potential markers
- **Type 4 (TFMarker):** TFs which play roles as markers

Download files (access 2025-11-17) (year/month/day)

These files were manually downloaded, because the server blocks direct automated download requests without a valid session and referer.

URLS:

- "data/marker_database/all_tf_markers.txt": "http://www.licpathway.net/TF-Marker/public/download/main_download.csv"
- "data/marker_database/tf_i_markers.txt": "http://www.licpathway.net/TF-Marker/public/download/download_I%20Marker.csv"
- "data/marker_database/tf_t_markers.txt": "http://www.licpathway.net/TF-Marker/public/download/download_T%20Marker.csv"
- "data/marker_database/tf_tf.txt": "http://www.licpathway.net/TF-Marker/public/download/download_TF.csv"
- "data/marker_database/tf_tf_pmarker.txt": "http://www.licpathway.net/TF-Marker/public/download/download_TF%20Pmarker.csv"
- "data/marker_database/tf_tf_marker": "http://www.licpathway.net/TF-Marker/public/download/download_TFMarker.csv"

### Load data

In [7]:
# Gene lists
panglao_df = pd.read_csv(here('data/marker_database/panglaodb_markers.tsv.gz'), sep='\t', compression='gzip')
tf_paths = glob.glob(os.path.join(markers_dir, "tf_*.txt"))

tf_df = {}
for file in tf_paths:
    name = Path(file).stem
    tf_df[name] = pd.read_csv(file, sep=",", dtype=str)
    
cellmarker_df = pd.read_excel(here('data/marker_database/cellmarker20.xlsx'), sheet_name='human')

# renameing map
reverse_map = pd.read_csv(os.path.join(markers_dir, 'cell_type_map.csv'))
reverse_map = reverse_map.set_index('old_cell_name')['new_cell_name'].to_dict()

### Make dictionary for dataframes

Panglao

In [8]:
panglao_df_flt = panglao_df[
    (panglao_df["organ"].isin(['Pancreas', 'Immune system', 'Vasculature'])) &
    (panglao_df['species'].isin(['Hs', 'Mm Hs'])) &
    (panglao_df['sensitivity_human'] > 0.5) &
    (panglao_df['specificity_human'] < 0.1)
].copy()

# Standardize cell type names using reverse_map
panglao_df_flt.rename(columns={'cell type': 'old_cell_name'}, inplace=True)
panglao_df_flt['cell_type_simple'] = panglao_df_flt['old_cell_name'].apply(
    lambda x: reverse_map.get(misc.to_snake_case(x), misc.to_snake_case(x))
)

# Make dictionary
panglao_genes = panglao_df_flt.groupby('cell_type_simple')['official gene symbol'].apply(list).to_dict()

In [9]:
cellmarker_df_flt = cellmarker_df[
    (cellmarker_df['tissue_type'].isin(['Pancreas', 'Pancreatic acinar tissue', 'Pancreatic duct'])) &
    (cellmarker_df['cancer_type'] == 'Normal') &
    (cellmarker_df['cell_type'] == 'Normal cell')].copy()

# Standardize cell type names using reverse_map
cellmarker_df_flt.rename(columns={'cell_name': 'old_cell_name'}, inplace=True)
cellmarker_df_flt['cell_type_simple'] = cellmarker_df_flt['old_cell_name'].apply(
    lambda x: reverse_map.get(misc.to_snake_case(x), misc.to_snake_case(x))
)

Cell Marker

In [10]:
cellmarker_df_flt = cellmarker_df[
    (cellmarker_df['tissue_type'].isin(['Pancreas', 'Pancreatic acinar tissue', 'Pancreatic duct'])) &
    (cellmarker_df['cancer_type'] == 'Normal') &
    (cellmarker_df['cell_type'] == 'Normal cell')].copy()

# Standardize cell type names using reverse_map
cellmarker_df_flt.rename(columns={'cell_name': 'old_cell_name'}, inplace=True)
cellmarker_df_flt['cell_type_simple'] = cellmarker_df_flt['old_cell_name'].apply(
    lambda x: reverse_map.get(misc.to_snake_case(x), misc.to_snake_case(x))
)

cellmarker_df_flt = (cellmarker_df_flt.dropna(subset=['cell_type_simple', 'Symbol', 'cellontology_id'])
                     .reset_index()[['cell_type_simple', 'marker', 'cellontology_id']]
                     .drop_duplicates())

cellmarker_df_flt.rename(columns={'cellontology_id': 'cell_ontology_id',
                                 'cell_type_simple': 'cell_type',
                                 'marker': 'gene'}, inplace=True)

TF df database

In [11]:
tf_df_pancreas = {}

for name, df in tf_df.items():
    df_flt = df.copy()[(df["Tissue Type"].isin(["Pancreas"]) &
                        df["Cell Type"].isin(["Normal cell"]))] 
    df_flt .rename(columns={'Cell Name': 'old_cell_name'}, inplace=True)
    df_flt['cell_type_simple'] = df_flt ['old_cell_name'].apply(
        lambda x: reverse_map.get(misc.to_snake_case(x), misc.to_snake_case(x))
    )

    df_flt = (df_flt.dropna(subset=['cell_type_simple', 'Gene Name', 'CellOntologyID'])
                     .reset_index()[['cell_type_simple', 'Gene Name', 'CellOntologyID']]
                     .drop_duplicates())

    df_flt.rename(columns={'CellOntologyID': 'cell_ontology_id',
                          'cell_type_simple': 'cell_type',
                          'Gene Name': 'gene'}, inplace=True)
    
    tf_df_pancreas[name] = df_flt

Save genelists

In [12]:
# Azimuth
rows = []
for celltype, genes in azimuth_genes.items():
    for gene in genes:
        rows.append({"cell_type": celltype, "gene": gene})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(harmo_dir, 'azimuth_genes.csv'), index=False)

# Panglao
rows = []
for celltype, genes in panglao_genes.items():
    for gene in genes:
        rows.append({"cell_type": celltype, "gene": gene})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(harmo_dir, 'panglao_genes.csv'), index=False)

# Cellmarker
cellmarker_df_flt.to_csv(os.path.join(harmo_dir, 'cellmarker_genes.csv'), index=False)

# TF database
for name, df in tf_df_pancreas.items():
    df.to_csv(os.path.join(harmo_dir, f'{name}_genes.csv'), index=False)

How to load genelists again

In [13]:
# path to genelists
genelists_path = glob.glob(os.path.join(harmo_dir, '*.csv'))

# Load and convert to dictionary
genelists = {}
for file in genelists_path:
    name = Path(file).stem.replace('_genes', '')
    df = pd.read_csv(file, sep=",", dtype=str)
    genelists[name] = df.groupby('cell_type')['gene'].apply(list).to_dict()